In [1]:
# 1. Paramètres et connexion
import sys
sys.path.append("../pipelines")
from aq import *            # eng(), reader(), lire(), BATCH, DATE_REF...
import pandas as pd

SOURCES = ["cif", "crm"]
print("Lot :", BATCH, "| date de référence :", DATE_REF)
print(lire("SELECT VERSION() AS version, @@collation_server AS collation"))

Lot : L-2026-10-01 | date de référence : 2026-10-01
                  version          collation
0  11.2.6-MariaDB-ubu2204  utf8mb4_nopad_bin


In [2]:
# 2. Inventaire : tables, colonnes, clé primaire, lignes estimées et lignes exactes
tables = lire("""SELECT table_schema AS base, table_name AS `table`, table_rows AS lignes_estimees
                  FROM information_schema.tables
                  WHERE table_schema IN ('cif','crm') AND table_type = 'BASE TABLE'""")
cols = lire("""SELECT table_schema AS base, table_name AS `table`, COUNT(*) AS colonnes
                FROM information_schema.columns WHERE table_schema IN ('cif','crm')
                GROUP BY table_schema, table_name""")
cles = lire("""SELECT table_schema AS base, table_name AS `table`,
                       GROUP_CONCAT(column_name ORDER BY ordinal_position) AS cle_primaire
                FROM information_schema.key_column_usage
                WHERE table_schema IN ('cif','crm') AND constraint_name = 'PRIMARY'
                GROUP BY table_schema, table_name""")

inventaire = tables.merge(cols, on=["base", "table"]).merge(cles, how="left", on=["base", "table"])
inventaire["lignes"] = [int(lire(f"SELECT COUNT(*) AS n FROM {b}.{t}").n[0])
                        for b, t in zip(inventaire.base, inventaire.table)]
inventaire["ecart_estimation"] = inventaire.lignes - pd.to_numeric(inventaire.lignes_estimees).fillna(0).astype(int)
inventaire = inventaire.sort_values(["base", "table"]).reset_index(drop=True)
print(inventaire.to_string(index=False))

print("\nTables sans clé primaire :", inventaire[inventaire.cle_primaire.isna()][["base", "table"]].values.tolist() or "aucune")
print("Total des lignes sources :", f"{inventaire.lignes.sum():,}".replace(",", " "))

base        table  lignes_estimees  colonnes cle_primaire  lignes  ecart_estimation
 cif      clients            35933        10      column0   40000              4067
 cif       compte            45901         4      column0   45000              -901
 cif         pays              250         2    code_pays     250                 0
 crm       client           109819        13    client_id  100000             -9819
 crm      contrat            99747         9   contrat_id  100000               253
 crm     sinistre            99320        10  sinistre_id  100000               680
 crm type_contrat               12         3    code_type      12                 0

Tables sans clé primaire : aucune
Total des lignes sources : 385 262


In [3]:
# 3. Description des colonnes
colonnes = lire("""SELECT table_schema AS base, table_name AS `table`, ordinal_position AS rang,
                           column_name AS colonne, column_type AS type, is_nullable AS nullable, column_key AS cle
                    FROM information_schema.columns WHERE table_schema IN ('cif','crm')
                    ORDER BY table_schema, table_name, ordinal_position""")
print(colonnes.to_string(index=False))

# Répartition des types déclarés par table : combien de colonnes texte, date, numériques ?
colonnes["famille"] = colonnes.type.str.extract(r"^(varchar|char|date|int|decimal)", expand=False).map(
    {"varchar": "texte", "char": "texte", "date": "date", "int": "numérique", "decimal": "numérique"}).fillna("autre")
print("\n", colonnes.pivot_table(index=["base", "table"], columns="famille", values="colonne", aggfunc="count", fill_value=0))

base        table  rang                  colonne          type nullable cle
 cif      clients     1                  column0   varchar(10)       NO PRI
 cif      clients     2                  column1  varchar(120)      YES    
 cif      clients     3                  column2   varchar(80)      YES    
 cif      clients     4                  column3   varchar(10)      YES    
 cif      clients     5                  column4  varchar(120)      YES    
 cif      clients     6                  column5   varchar(10)      YES    
 cif      clients     7                  column6   varchar(80)      YES    
 cif      clients     8                  column7    varchar(5)      YES    
 cif      clients     9                  column8   varchar(25)      YES    
 cif      clients    10                  column9  varchar(120)      YES    
 cif       compte     1                  column0   varchar(12)       NO PRI
 cif       compte     2                  column1   varchar(10)      YES MUL
 cif       c

In [5]:
# 4. NULL / vide / blanc par colonne, pour les sept tables
donnees = {}
lignes_detail = []
for b, t in zip(inventaire.base, inventaire.table):
    df = lire(f"SELECT * FROM {t}", reader(b)).astype("string")
    donnees[f"{b}.{t}"] = df
    for c in df.columns:
        s = df[c]
        n_null = int(s.isna().sum())
        n_vide = int((s == "").sum())
        n_blanc = int(((s.str.strip() == "") & (s != "")).sum())
        lignes_detail.append({"table": f"{b}.{t}", "colonne": c, "lignes": len(df), "null": n_null, "vide": n_vide,
                              "blanc": n_blanc, "pct_manquant": round(100 * (n_null + n_vide + n_blanc) / len(df), 2),
                              "distincts": int(s.nunique())})
detail = pd.DataFrame(lignes_detail)
print(detail[detail.pct_manquant > 0].sort_values(["table", "pct_manquant"], ascending=[True, False]).to_string(index=False))

par_table = detail.groupby("table").agg(colonnes=("colonne", "count"),
                                        colonnes_incompletes=("pct_manquant", lambda x: int((x > 0).sum())),
                                        null=("null", "sum"), vide=("vide", "sum"), blanc=("blanc", "sum")).reset_index()
print("\nSynthèse par table :\n", par_table.to_string(index=False))

       table                  colonne  lignes  null  vide  blanc  pct_manquant  distincts
 cif.clients                  column9   40000 18872     0      0         47.18      21128
 cif.clients                  column8   40000  6324     0      0         15.81      33676
 cif.clients                  column2   40000   645     0      0          1.61        180
 cif.clients                  column3   40000   645     0      0          1.61      21081
  crm.client           raison_sociale  100000 95723     0      0         95.72       3229
  crm.client                    email  100000 19365     0      0         19.36      77795
  crm.client                telephone  100000  5999  2000   2000         10.00      88702
  crm.client                      nom  100000  4277     0      0          4.28        784
  crm.client                   prenom  100000  4277     0      0          4.28        293
  crm.client           date_naissance  100000  4277     0      0          4.28      28413
  crm.clie

In [6]:
# 5.1 Unicité et absence de NULL sur chaque clé primaire (vérification par requête)
verif_cles = []
for b, t, pk in inventaire[["base", "table", "cle_primaire"]].itertuples(index=False):
    r = lire(f"SELECT COUNT(*) AS n, COUNT(DISTINCT {pk}) AS d, SUM({pk} IS NULL) AS nuls FROM {b}.{t}")
    verif_cles.append({"table": f"{b}.{t}", "cle": pk, "lignes": int(r.n[0]), "valeurs_distinctes": int(r.d[0]),
                       "doublons": int(r.n[0]) - int(r.d[0]), "nuls": int(r.nuls[0] or 0)})
print(pd.DataFrame(verif_cles).to_string(index=False))

# 5.2 Clés étrangères déclarées dans les sources
fk = lire("""SELECT table_schema AS base, table_name AS `table`, column_name AS colonne,
                     referenced_table_schema AS base_ref, referenced_table_name AS table_ref
              FROM information_schema.key_column_usage
              WHERE table_schema IN ('cif','crm') AND referenced_table_name IS NOT NULL""")
print("\nClés étrangères déclarées :", len(fk))
print(fk.to_string(index=False) if len(fk) else "aucune : l'intégrité entre tables n'est pas garantie par le SGBD (à mesurer en S3)")

           table         cle  lignes  valeurs_distinctes  doublons  nuls
     cif.clients     column0   40000               40000         0     0
      cif.compte     column0   45000               45000         0     0
        cif.pays   code_pays     250                 250         0     0
      crm.client   client_id  100000              100000         0     0
     crm.contrat  contrat_id  100000              100000         0     0
    crm.sinistre sinistre_id  100000              100000         0     0
crm.type_contrat   code_type      12                  12         0     0

Clés étrangères déclarées : 0
aucune : l'intégrité entre tables n'est pas garantie par le SGBD (à mesurer en S3)


In [11]:
D = donnees

cl = D["crm.client"]
ct = D["crm.contrat"]
sn = D["crm.sinistre"]
tc = D["crm.type_contrat"]
cc = D["cif.clients"]
cp = D["cif.compte"]
py = D["cif.pays"]

duree = pd.to_numeric(ct.duree_garantie_mois, errors="coerce")
ref = set(lire("SELECT code_pays FROM silver.ref_pays").code_pays)

chiffres = pd.DataFrame([
    ("crm.client", "personnes morales", int(cl.raison_sociale.notna().sum())),
    ("crm.client", "ni nom ni raison sociale", int((cl.nom.isna() & cl.raison_sociale.isna()).sum())),
    ("crm.client", "nom et raison sociale", int((cl.nom.notna() & cl.raison_sociale.notna()).sum())),
    ("crm.contrat", "durée hors {12, 24, 36, 60}", int((duree.notna() & ~duree.isin([12, 24, 36, 60])).sum())),
    ("crm.contrat", "date de fin < date de début", int((pd.to_datetime(ct.date_fin, errors="coerce") < pd.to_datetime(ct.date_debut, errors="coerce")).sum())),
    ("crm.sinistre", "déclaration < survenance", int((pd.to_datetime(sn.date_declaration, errors="coerce") < pd.to_datetime(sn.date_survenance, errors="coerce")).sum())),
    ("crm.sinistre", "montant <= 0", int((pd.to_numeric(sn.montant_estime, errors="coerce") <= 0).sum())),
    ("crm.sinistre", "canaux de saisie distincts", int(sn.canal_saisie.nunique())),
    ("cif.clients", "colonnes de type texte", int((colonnes[(colonnes.base == "cif") & (colonnes.table == "clients")].famille == "texte").sum())),
    ("cif.compte", "% column1 présents dans cif.clients.column0", round(100 * cp.column1.isin(set(cc.column0)).mean(), 2)),
    ("cif.compte", "column1 NULL", int(cp.column1.isna().sum())),
    ("cif.pays", "codes pays", int(py.code_pays.nunique())),
    ("cif.pays", "codes présents dans silver.ref_pays", int(py.code_pays.isin(ref).sum())),
], columns=["table", "mesure", "valeur"])

print(chiffres.to_string(index=False))
print(tc.branche.value_counts().to_string())

       table                                      mesure  valeur
  crm.client                           personnes morales  4277.0
  crm.client                    ni nom ni raison sociale     0.0
  crm.client                       nom et raison sociale     0.0
 crm.contrat                 durée hors {12, 24, 36, 60}   600.0
 crm.contrat                 date de fin < date de début   400.0
crm.sinistre                    déclaration < survenance     0.0
crm.sinistre                                montant <= 0     0.0
crm.sinistre                  canaux de saisie distincts     4.0
 cif.clients                      colonnes de type texte    10.0
  cif.compte % column1 présents dans cif.clients.column0   100.0
  cif.compte                                column1 NULL     0.0
    cif.pays                                  codes pays   250.0
    cif.pays         codes présents dans silver.ref_pays   250.0
branche
AUTO          4
HABITATION    4
SANTE         4


In [12]:
import os
from IPython.display import Markdown, display

fmt = lambda n: f"{int(n):,}".replace(",", " ")

tab = inventaire.sort_values(["base", "table"], ascending=[False, True])
lignes_md = ["| Source | Table | Lignes | Clé primaire | Observation chiffrée |",
             "|---|---|---|---|---|"]
for r in tab.itertuples(index=False):
    lignes_md.append(f"| {r.base} | {r.table} | {fmt(r.lignes)} | {r.cle_primaire} | … |")
md = "\n".join(lignes_md)

print(md)
display(Markdown(md))

dossier = "/home/jovyan/work/reports" if os.path.isdir("/home/jovyan/work/reports") else "../reports"
with open(f"{dossier}/S1_US01_inventaire.md", "w", encoding="utf-8") as f:
    f.write(md + "\n")

aide = par_table.merge(pd.DataFrame(verif_cles)[["table", "doublons", "nuls"]], on="table")
aide = aide.rename(columns={"colonnes_incompletes": "colonnes_avec_manques", "doublons": "doublons_cle", "nuls": "cle_nulle"})
print(aide.to_string(index=False))
print(detail[detail.pct_manquant > 0][["table", "colonne", "null", "vide", "blanc", "pct_manquant"]].to_string(index=False))

| Source | Table | Lignes | Clé primaire | Observation chiffrée |
|---|---|---|---|---|
| crm | client | 100 000 | client_id | … |
| crm | contrat | 100 000 | contrat_id | … |
| crm | sinistre | 100 000 | sinistre_id | … |
| crm | type_contrat | 12 | code_type | … |
| cif | clients | 40 000 | column0 | … |
| cif | compte | 45 000 | column0 | … |
| cif | pays | 250 | code_pays | … |


| Source | Table | Lignes | Clé primaire | Observation chiffrée |
|---|---|---|---|---|
| crm | client | 100 000 | client_id | … |
| crm | contrat | 100 000 | contrat_id | … |
| crm | sinistre | 100 000 | sinistre_id | … |
| crm | type_contrat | 12 | code_type | … |
| cif | clients | 40 000 | column0 | … |
| cif | compte | 45 000 | column0 | … |
| cif | pays | 250 | code_pays | … |

           table  colonnes  colonnes_avec_manques   null  vide  blanc  doublons_cle  cle_nulle
     cif.clients        10                      4  26486     0      0             0          0
      cif.compte         4                      0      0     0      0             0          0
        cif.pays         2                      0      0     0      0             0          0
      crm.client        13                      7 133918  2400   2000             0          0
     crm.contrat         9                      1   7460     0      0             0          0
    crm.sinistre        10                      1      0  1965      0             0          0
crm.type_contrat         3                      0      0     0      0             0          0
       table                  colonne  null  vide  blanc  pct_manquant
 cif.clients                  column2   645     0      0          1.61
 cif.clients                  column3   645     0      0          1.61
 cif.clients               